# File Handling (Text, Binary, CSV, JSON)


# File Handling

It allows you to permanently store, read, modify, and delete data on a storage device using the built-in open() function. 

---

### The Context Manager (`with` Statement)

It acts as a context manager, meaning it **automatically closes the file** after your code block executes. This prevents data loss, memory leaks, and locked system resources.

In [1]:
filePath = "./files/file.txt"

# The safest and most common way to handle files
with open(filePath, "r") as file:
    content = file.read()
# File automatically closes here

---

# Core File Access Methods

The `open(filename, mode)` function takes an access mode parameter to dictate what operations are permitted:

| Mode | Purpose | Description |
| --- | --- | --- |
| `"r"` | Read (default) | Opens a file for reading; raises an error if it doesn't exist. | 
| `"w"` | Write | Creates or overwrites a file; completely wipes existing content. |
| `"a"` | Append | Adds data to the end of an existing file; creates it if missing. |
| `"x"` | Create | Exclusive creation; fails and throws an error if the file exists. |
| `"b"` | Binary | Used for non-text formats like images, videos, or PDFs (e.g., `"rb"`, `"wb"`) |

---

# Reading Methods

Python provides three direct methods to retrieve data from text files:

- **`readline()`**: Reads exactly one single line at a time from the file.

- **`readlines()`**: Reads all lines globally and parses them into a list of strings.

- **`for line in file:`** Iterates through lines sequentially; best choice for processing massive files.

In [2]:
with open(filePath, "r", encoding="Utf-8") as file:
    for line in file:
        print(line)

Hello, world

Line 2

Line 3



> Notice: There is newline `\n` that creating space between the texts. So, we will use `strip()` to remove trailing newlines.

In [3]:
with open(filePath, "r", encoding="utf-8") as file:
    for line in file:
        print(line.strip())  # strip() removes trailing newlines (\n)

Hello, world
Line 2
Line 3


> Notice the difference between earlier output and this output or with `.strip()` and without it.

---

# Writing and Appending Methods

To put data into a text file, open it in "w" or "a" mode and use one of these:

- **`write(string)`**: Inserts a raw string explicitly into the document file.

- **`writelines(list)`**: Automatically unrolls a list of strings sequentially into the file.

In [4]:
# Overwriting content
with open(filePath, "w") as file:
    file.write("Hello, world\n")

# confirm after write
with open(filePath, "r", encoding="utf-8") as file:
    for line in file:
        print(line.strip())

Hello, world


In [5]:
# Appending content safely
with open(filePath, "a") as file:
    file.writelines(["Line 2\n", "Line 3\n"])

# confirm after appending content
with open(filePath, "r", encoding="utf-8") as file:
    for line in file:
        print(line.strip())

Hello, world
Line 2
Line 3


---

# Exception Handling

Files can throw unexpected runtime errors if they are missing or if permissions are denied. Use a `try-except` block to build fault-tolerant scripts.

In [6]:
try:
    with open(filePath, "r") as file:
        data = file.read()
except FileNotFoundError:
    print("Error: The requested file could not be found.")
except PermissionError:
    print("Error: You do not have permissions to read this file.")

# Binary File Handling

**Core File Modes:**

- **`"wb" (write binary)`**: Writes new binary data, completely overwriting existing data.

- **`"rb" (read binary)`**: Reads existing binary data.

- **`"ab" (append binary)`**: Appends data to the end of an existing binary file.

---

# Writing to a Binary File

When writing, your data source must be a `bytes`, `bytearray`, or derived binary object. Strings must be explicitly encoded.

In [4]:
# filename
filename = "./files/file.bin"
filename2 = "./files/file2.bin"

# Raw byte string literal (Note the 'b' prefix)
binary_data = b"Hello, this is binary data!"

# Writing bytes to a file
with open(filename, "wb") as file:
    file.write(binary_data)

In [6]:
# Converting a regular string to bytes before writing
text_str = "Convert me to bytes"

with open(filename2, "wb") as file:
    file.write(text_str.encode("utf-8"))

---

# Reading a Binary File

Reading a binary file yields a bytes object. If the file contains text data, you must decode it back into a string.

In [7]:
# Reading the entire file at once
with open(filename, "rb") as file:
    content = file.read()
    print(content)

b'Hello, this is binary data!'


In [8]:
# Decoding if the binary data represents text

with open(filename2, "rb") as file:
    raw_bytes = file.read()
    text = raw_bytes.decode("utf-8")
    print(text)

Convert me to bytes


---

# Processing Large Binary Files (Chunking)

Loading giant binary files (like videos or databases) entirely into your system's memory can cause a crash. Read them in designated chunk sizes instead.

In [ ]:
chunk_size = 4096  # Read 4kb  at a time

with open("large_file.bin", "rb") as file:
    while True:
        chunk = file.read(chunk_size)
        if not chunk:
            break

---

# Working with Python Objects (Serialization)

If you want to save actual Python objects (like lists, dictionaries, or custom data structures) directly to a binary file, use the built-in `pickle` module.

In [1]:
import pickle
import uuid

data_structure = {"user_id": str(uuid.uuid4()), "scores": [100, 95, 98]}

# Saving the object to a binary file
with open("./files/dataset.pkl", "wb") as file:
    pickle.dump(data_structure, file)

In [2]:
# Loading the object back into Python
with open("./files/dataset.pkl", "rb") as file:
    loaded_data = pickle.load(file)
    print(loaded_data)

{'user_id': 'ddecc175-babf-496c-b463-778e54721b4a', 'scores': [100, 95, 98]}


---

# Packing Structured C-Type Binary Data

For low-level binary tasks (like reading custom file headers or specific numeric formats), use the `struct` module to pack variables into a rigid format.

In [4]:
import struct

# Pack an integer (i) and a float (f) into binary format
binary_packet = struct.pack("if", 1024, 3.14)

with open("./files/structured.bin", "wb") as file:
    file.write(binary_packet)

In [5]:
# Read and unpack the structured binary data
with open("./files/structured.bin", "rb") as file:
    raw_data = file.read()
    unpacked_data = struct.unpack("if", raw_data)
    print(unpacked_data)

(1024, 3.140000104904175)


# `csv` Module

Python handles CSV files primarily using the built-in `csv` module, which provides tools to read and write comma-separated values easily.

---

# Writing CSV Files

- **Standard Writing**: Uses `csv.writer()` to write rows using lists.

- **Dictionary Writing**: Uses `csv.DictWriter()` to write rows using dictionaries and handles headers.

### Using standard writer

In [ ]:
import csv

data = [
    ["Name", "Age", "City"],
    ["Ali", "24", "Rawalpindi"],
    ["Hamza", "23", "Kohat"],
    ["Abdullah", "22", "Sialkot"],
]

with open("./files/output.csv", mode="w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerows(data)  # write multiple rows

> `utf-8 | utf-16 | utf-32` (Unicode Transformation Format): are character encodings that translate human-readable text into binary data (0s and 1s) so computers can store and transmit it.

### Using dictionary writer

In [3]:
import csv

fieldnames = ["Name", "Age", "City"]  # header schema
rows = [
    {"Name": "Ali", "Age": "24", "City": "Rawalpindi"},
    {"Name": "Hamza", "Age": "23", "City": "Kohat"},
    {"Name": "Abdullah", "Age": "22", "City": "Sialkot"},
]

with open("./files/output_dict.csv", mode="w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()  # writes the top header
    writer.writerows(rows)

---

# Reading CSV Files

- **Standard Reading**: Uses `csv.reader()` to parse rows into lists of strings.

- **Dictionary Reading**: Uses `csv.DictReader()` to map headers directly to row values.

### Using standard reader

In [ ]:
import csv

with open("./files/output.csv", mode="r", newline="", encoding="utf-8") as file:
    reader = csv.reader(file)
    for row in reader:
        print(row)  # Each row is a list of strings

['Name', 'Age', 'City']
['Ali', '24', 'Rawalpindi']
['Hamza', '23', 'Kohat']
['Abdullah', '22', 'Sialkot']


### Using dictionary reader

In [7]:
import csv

with open("./files/output_dict.csv", mode="r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    for row in reader:
        print(row["Name"], row["Age"], row["City"])

Ali 24 Rawalpindi
Hamza 23 Kohat
Abdullah 22 Sialkot


# `json` Module

The Python `json` module is a built-in library used to read, write, and manipulate JSON (JavaScript Object Notation) data.

### Four Core Functions

The entire module revolves around four primary functions. The names tell you exactly what they do if you remember one rule: Functions ending in "s" handle Strings, while functions without "s" handle File objects.

- **`json.loads()`**: Loads a JSON String into a Python object.

- **`json.load()`**: Loads JSON data from a File object into a Python object.

- **`json.dumps()`**: Dumps a Python object into a JSON String.

- **`json.dump()`**: Dumps a Python object directly into a File object.

---

# 1. Converting JSON Strings to Python (`loads`)

When you fetch data from a web API, it usually arrives as a raw text string (json-format). You use json.loads() to convert that string into a native Python dictionary or list

In [11]:
import json

# A raw string containing JSON data
json_string = '{"name": "Alice", "age": 25, "is_student": false, "skills": ["Python", "SQL"]}'

# Convert the string into a Python dictionary
data = json.loads(json_string)

print(f"{'Index':<10} | {'Field':<15} | {'Values'}")
print("-" * 48)
for ind, (k, v) in enumerate(data.items()):
    print(f"{ind:<10} | {k:<15} | {v}")

Index      | Field           | Values
------------------------------------------------
0          | name            | Alice
1          | age             | 25
2          | is_student      | False
3          | skills          | ['Python', 'SQL']


---

# 2. Converting Python to JSON Strings (`dumps`)

When you want to prepare data to send to an API or another application, you must convert your Python dictionary back into a raw text string.

In [18]:
import json

user_dict = {"name": "Bob", "age": 30, "has_license": True, "hobbies": None}

# Convert the dictionary into a JSON formatted string
raw_json = json.dumps(user_dict)

print(type(raw_json))
print(raw_json)

<class 'str'>
{"name": "Bob", "age": 30, "has_license": true, "hobbies": null}


---

# 3. Reading and Writing JSON Files (load and dump)

### Writing to a File (`dump`)

In [20]:
import json

payload = {"name": "Bob", "age": 30, "has_license": True, "hobbies": None}

with open("./files/payload.json", mode="w") as file:
    json.dump(payload, file, indent=4)  # Saves directly into the file

### Reading from a File (`load`)

In [21]:
import json

with open("./files/payload.json", mode="r") as file:
    payload = json.load(file)

print(payload)

{'name': 'Bob', 'age': 30, 'has_license': True, 'hobbies': None}
